In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from collections import Counter
from nltk.tokenize import word_tokenize
import nltk
import json
import torch.optim as optim
from sklearn.metrics import f1_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
df_train = pd.read_csv('../data/train_lemmastop.csv')
df_train['processed_text'] = df_train['processed_text'].fillna('')
df_train.head()

,id,text,anger,fear,joy,sadness,surprise,emotions,clean_text,processed_text
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness'],the dentist that did the work apparently did a...,dentist work apparently lousy job year teeth d...
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness'],i am going to absolutely be terrible during my...,going absolutely terrible first sexual experience
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness'],bridge so leave me drowning calling houston an...,bridge leave drowning calling houston let lung...
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness'],after that mess i went to see my now exgirlfri...,mess went see exgirlfriend school refused driv...
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear'],as he stumbled i ran off afraid it might someh...,stumbled ran afraid might somehow affect job s...


In [4]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']
labels = df_train[emotion_cols].values

In [5]:
all_texts = ' '.join(df_train['processed_text'])
words = word_tokenize(all_texts)
word_counts = Counter(words)

In [6]:
len(word_counts)

7458

In [10]:
# Create the vocabulary
# Start with our special tokens
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.items():
    if count > 1:
        vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)} unique words")

Vocabulary size: 4873 unique words


In [11]:
with open('../data/LSTM_Model_vocab.json', 'w') as f:
    json.dump(vocab, f)

In [12]:
def text_to_sequence(text, vocab):
    """Converts a text string to a sequence of integers using the vocab."""
    tokens = word_tokenize(text)
    return [vocab.get(word, 1) for word in tokens]

In [13]:
sequences = [text_to_sequence(text, vocab) for text in df_train['processed_text']]
seq_lengths = [len(s) for s in sequences]
MAX_LEN = int(np.percentile(seq_lengths, 95))
print(f"Using a max sequence length of: {MAX_LEN}")

Using a max sequence length of: 18


In [14]:
def pad_sequence(seq, max_len):
    """Pads a sequence to max_len. Truncates if longer."""
    if len(seq) > max_len:
        return seq[:max_len]  # Truncate
    else:
        return seq + [0] * (max_len - len(seq))

In [15]:
padded_sequences = [pad_sequence(s, MAX_LEN) for s in sequences]

X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32)
print(f"Shape of features (X): {X.shape}")
print(f"Shape of labels (y): {y.shape}")

Shape of features (X): torch.Size([6827, 18])
Shape of labels (y): torch.Size([6827, 5])


In [16]:
class EmotionDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [17]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
train_dataset = EmotionDataset(X_train, y_train)
val_dataset = EmotionDataset(X_val, y_val)

In [19]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders created successfully.")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

DataLoaders created successfully.
Train batches: 171
Validation batches: 43


In [22]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers, 
            bidirectional=True, 
            dropout=dropout,
            batch_first=True
        )
        
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text shape: [batch_size, seq_len]
        embedded = self.embedding(text)
        # embedded shape: [batch_size, seq_len, embedding_dim]
        
        # outputs shape: [batch_size, seq_len, hidden_dim * 2]
        # hidden shape: [n_layers * 2, batch_size, hidden_dim]
        outputs, (hidden, cell) = self.lstm(embedded)
        
        # Concatenate the final forward and backward hidden states
        # hidden[-2,:,:] is the final forward hidden state
        # hidden[-1,:,:] is the final backward hidden state
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        # hidden shape: [batch_size, hidden_dim * 2]

        hidden = self.dropout(hidden)
        prediction = self.fc(hidden)
        # prediction shape: [batch_size, output_dim]
        
        return prediction

In [37]:
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100 
HIDDEN_DIM = 128     
OUTPUT_DIM = 5      
N_LAYERS = 2         
DROPOUT = 0.4        
LEARNING_RATE = 1e-3
N_EPOCHS = 50

In [38]:
model = LSTMModel(
    VOCAB_SIZE, 
    EMBEDDING_DIM, 
    HIDDEN_DIM, 
    OUTPUT_DIM, 
    N_LAYERS, 
    DROPOUT
)
model = model.to(device)  

In [39]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [40]:
def calculate_f1(preds, y_true, threshold=0.5):
    y_pred = torch.sigmoid(preds)
    y_pred = (y_pred > threshold).int()
    
    y_pred_cpu = y_pred.cpu().numpy()
    y_true_cpu = y_true.cpu().numpy()
    
    return f1_score(y_true_cpu, y_pred_cpu, average='macro', zero_division=0)

In [41]:
def train_epoch(model, iterator, optimizer, criterion):
    model.train() 
    epoch_loss = 0
    
    for features, labels in iterator:
        features = features.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()

        predictions = model(features)

        loss = criterion(predictions, labels)

        loss.backward()

        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

In [42]:
def evaluate_epoch(model, iterator, criterion):
    model.eval()  
    epoch_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for features, labels in iterator:
            features = features.to(device)
            labels = labels.to(device)
            
            predictions = model(features)
            loss = criterion(predictions, labels)
            
            epoch_loss += loss.item()
            
            all_preds.append(predictions)
            all_targets.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    f1 = calculate_f1(all_preds, all_targets)
    
    return epoch_loss / len(iterator), f1

In [45]:
print("Starting training...")

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1 = evaluate_epoch(model, val_loader, criterion)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch: {epoch+1:02}')
        print(f'\tTrain Loss: {train_loss:.3f}')
        print(f'\t Val. Loss: {val_loss:.3f} |  Val. Macro F1: {val_f1:.4f}')

print("Training finished.")

Starting training...
Epoch: 05
	Train Loss: 0.005
	 Val. Loss: 1.238 |  Val. Macro F1: 0.7084
Epoch: 10
	Train Loss: 0.005
	 Val. Loss: 1.283 |  Val. Macro F1: 0.7022
Epoch: 15
	Train Loss: 0.008
	 Val. Loss: 1.188 |  Val. Macro F1: 0.7033
Epoch: 20
	Train Loss: 0.006
	 Val. Loss: 1.150 |  Val. Macro F1: 0.7169
Epoch: 25
	Train Loss: 0.005
	 Val. Loss: 1.263 |  Val. Macro F1: 0.7120
Epoch: 30
	Train Loss: 0.011
	 Val. Loss: 1.184 |  Val. Macro F1: 0.7095
Epoch: 35
	Train Loss: 0.005
	 Val. Loss: 1.277 |  Val. Macro F1: 0.7067
Epoch: 40
	Train Loss: 0.008
	 Val. Loss: 1.177 |  Val. Macro F1: 0.7093
Epoch: 45
	Train Loss: 0.005
	 Val. Loss: 1.236 |  Val. Macro F1: 0.7120
Epoch: 50
	Train Loss: 0.005
	 Val. Loss: 1.258 |  Val. Macro F1: 0.7087
Training finished.
